# Cochleogram-ViT — Leak-Fixed + DOUBLE Imbalance Correction, SOFTEN_POWER=0.25

Builds on notebook 07 (leak-free Subset pipeline, paper-convention metric,
from-scratch ViT). Uses BOTH imbalance corrections at once:
- `WeightedRandomSampler` (correctly scoped via Subset -> no leakage), AND
- class-weighted `CrossEntropyLoss`.

Same configuration as notebook 05, but with the corrected sampler scoping and the
paper-convention metric, so it's directly comparable to 07 and 12.

Completes the imbalance ablation (all paper-convention):
  - 13 (this): sampler + weighted loss (double)
  - 07:        weighted loss only
  - 05:        double, but old metric (Sp collapsed)

Open question: does double correction still over-correct (Se up, Sp down) the way
05 did, or does it look different under the paper metric?


In [10]:
from cochleogram_vit.models.vit import CochleogramViT
import torch

# Instantiate the baseline ViT (no KAN).
vit_model = CochleogramViT(
    image_size=128, patch_size=16, num_classes=4, dim=512,
    depth=6, heads=8, mlp_dim=1024, channels=3,
)

# Shape smoke test
x = torch.randn(2, 3, 128, 128)
y = vit_model(x)
assert y.shape == (2, 4), f"unexpected output shape {y.shape}"
print("CochleogramViT smoke test OK — output shape", tuple(y.shape))


[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644
CochleogramViT smoke test OK — output shape (2, 4)


## 2. Forward Pass Test

Create a dummy batch of tensors and pass it through the model to ensure the input and output dimensions are correct.


In [11]:
# Create a dummy batch of 4 RGB cochleograms (Batch, Channels, Height, Width)
dummy_batch = torch.randn(4, 3, 128, 128) # Channels set to 3

# Perform a forward pass
with torch.no_grad():
    logits = vit_model(dummy_batch)

print(f"Input shape:  {dummy_batch.shape}")
print(f"Output shape: {logits.shape}")

# Check that the output shape is as expected (Batch, Num_Classes)
assert logits.shape == (4, 4)
print("\nSuccess! The model produced the correct output shape for 3-channel input.")


Input shape:  torch.Size([4, 3, 128, 128])
Output shape: torch.Size([4, 4])

Success! The model produced the correct output shape for 3-channel input.


## 3. Load Configuration and Data

Now, let's load the dataset. We will use the `ICBHIDataset` class and the patient-wise split function from your `src` directory to prepare for training.


In [12]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import matplotlib.pyplot as plt
import torch

# Configuration
DATA_DIR = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0001

# Custom Dataset
class CochleogramDataset(Dataset):
    def __init__(self, data_dir, metadata_path, transform=None):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        cochleogram = np.load(npy_path)  # already [0,1], no need to renormalize
        label = int(row['label'])

        # Apply Viridis colormap
        viridis_cmap = plt.get_cmap('viridis')
        colored_cochleogram = viridis_cmap(cochleogram)

        # Drop alpha, transpose to (C, H, W), make contiguous
        rgb_cochleogram = np.ascontiguousarray(colored_cochleogram[:, :, :3].transpose(2, 0, 1))
        cochleogram_tensor = torch.from_numpy(rgb_cochleogram).float()

        if self.transform:
            cochleogram_tensor = self.transform(cochleogram_tensor)

        return cochleogram_tensor, label

# Create dataset
dataset = CochleogramDataset(DATA_DIR, METADATA_PATH)

print(f"Dataset size: {len(dataset)}")
sample_img, sample_label = dataset[0]
print(f"Sample image shape: {sample_img.shape}")  # Should be (3, 128, 128)
print(f"Min: {sample_img.min():.4f}, Max: {sample_img.max():.4f}")
print(f"Any NaN: {torch.isnan(sample_img).any()}")
print(f"Any Inf: {torch.isinf(sample_img).any()}")

Dataset size: 6898
Sample image shape: torch.Size([3, 128, 128])
Min: 0.0049, Max: 0.8719
Any NaN: False
Any Inf: False


## 4. Train the Model

Now we'll set up the optimizer and loss function and run a basic training loop.


In [13]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm.auto import tqdm
from cochleogram_vit.models.vit import CochleogramViT
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import numpy as np
import copy

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Reproducibility ---
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# --- Cross-Validation Setup ---
metadata = dataset.metadata
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])
groups = metadata['patient_id'].values
gkf = GroupKFold(n_splits=10)

# --- Class Weights (softened with power 0.75) ---
raw_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2, 3]),
    y=metadata['label'].values
)
SOFTEN_POWER = 0.25  # class-weight softening exponent (0=uniform, 0.75=prev, 1=raw balanced)
class_weights = raw_weights ** SOFTEN_POWER
class_weights = class_weights / class_weights.sum() * len(class_weights)  # renormalize

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nClass weights (softened ^{SOFTEN_POWER}, renormalized):")
print(f"  Normal   (0): {class_weights[0]:.4f}")
print(f"  Crackles (1): {class_weights[1]:.4f}")
print(f"  Wheezes  (2): {class_weights[2]:.4f}")
print(f"  Both     (3): {class_weights[3]:.4f}")

# --- LR Warmup + Cosine Decay ---
def lr_lambda(epoch):
    warmup_epochs = 4
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    denom = EPOCHS - warmup_epochs
    if denom == 0:
        return 0.0
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / denom))

# Store results
fold_results = []
all_preds_total = []
all_labels_total = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(metadata, groups=groups)):
    print('\n' + '='*60)
    print(f'FOLD {fold+1}/10')
    print('='*60)

    # Keep train_labels for the distribution printout below.
    train_labels = metadata['label'].values[train_idx]

    # --- LEAK FIX: scope each loader to its fold via Subset ---
    # Subset(dataset, idx) maps position i -> dataset[idx[i]], so training can
    # only ever see train_idx and validation only val_idx.
    train_subset = torch.utils.data.Subset(dataset, train_idx)
    val_subset   = torch.utils.data.Subset(dataset, val_idx)

    # --- DOUBLE imbalance correction: WeightedRandomSampler + class-weighted loss ---
    # Sampler correctly scoped: sample_weights[i] aligns with train_subset[i] =
    # dataset[train_idx[i]], and it indexes train_subset (NOT the full dataset) -> no
    # leakage. The loss ALSO uses class weights (criterion below) -> both active.
    sample_weights = torch.tensor([class_weights[l] for l in train_labels], dtype=torch.float)
    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True
    )
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=train_sampler)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

    print(f"  Train samples: {len(train_idx)} | Val samples: {len(val_idx)}")
    print(f"  Train class distribution: {dict(sorted(Counter(train_labels.tolist()).items()))}")

    # --- Fixed seed per fold for reproducibility ---
    torch.manual_seed(42 + fold)
    np.random.seed(42 + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42 + fold)

    # --- Re-initialize model and optimizer for each fold ---
    vit_model = CochleogramViT(
        image_size=128, patch_size=16, num_classes=4, dim=512,
        depth=6, heads=8, mlp_dim=1024, channels=3,
        dropout=0.3, emb_dropout=0.2
    ).to(device)

    optimizer = optim.Adam(vit_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # --- Training Loop ---
    print(f"\n  {'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'LR':<12} {'Status'}")
    print(f"  {'-'*60}")

    best_score = 0.0
    best_model_state = None
    best_epoch = 1

    for epoch in range(EPOCHS):
        # Training phase
        vit_model.train()
        running_loss = 0.0
        for cochleograms, labels in tqdm(train_loader, desc=f"  Epoch {epoch+1}/{EPOCHS}", leave=False):
            cochleograms, labels = cochleograms.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = vit_model(cochleograms)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        # Validation phase
        vit_model.eval()
        val_loss = 0.0
        val_preds = []
        val_labels_epoch = []
        with torch.no_grad():
            for cochleograms, labels in val_loader:
                cochleograms, labels = cochleograms.to(device), labels.to(device)
                outputs = vit_model(cochleograms)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_epoch.extend(labels.cpu().numpy())
        val_loss /= len(val_loader)

        # ── Per-epoch metric block (used for best checkpoint selection) ──
        val_preds_arr  = np.array(val_preds)
        val_labels_arr = np.array(val_labels_epoch)

        TP_e = np.sum((val_labels_arr != 0) & (val_preds_arr == val_labels_arr))
        FN_e = np.sum((val_labels_arr != 0) & (val_preds_arr == 0))                                                     # paper def: adventitious → normal
        FN_wrong_type_e = np.sum((val_labels_arr != 0) & (val_preds_arr != 0) & (val_preds_arr != val_labels_arr))     # subtype confusion, stored only
        TN_e = np.sum((val_labels_arr == 0) & (val_preds_arr == 0))
        FP_e = np.sum((val_labels_arr == 0) & (val_preds_arr != 0))

        assert FN_e + FN_wrong_type_e + TP_e == np.sum(val_labels_arr != 0), "Epoch adventitious decomposition mismatch"

        sensitivity_e = (TP_e + FN_wrong_type_e) / (TP_e + FN_wrong_type_e + FN_e + 1e-8)  # paper convention
        specificity_e = TN_e / (TN_e + FP_e + 1e-8)
        epoch_score   = (sensitivity_e + specificity_e) / 2.0

        # Save best checkpoint based on score
        if epoch_score > best_score:
            best_score = epoch_score
            best_model_state = copy.deepcopy(vit_model.state_dict())
            best_epoch = epoch + 1

        current_lr = optimizer.param_groups[0]['lr']

        # Per-epoch logging: loss curves + val Se/Sp/Score (diagnose convergence)
        marker = "  <- best" if best_epoch == epoch + 1 else ""
        print(f"  {epoch+1:<8} {train_loss:<14.4f} {val_loss:<14.4f} {current_lr:<12.2e} "
              f"Se={sensitivity_e*100:5.1f} Sp={specificity_e*100:5.1f} Score={epoch_score*100:5.1f}{marker}")

        scheduler.step()

    print(f"\n  Best checkpoint at epoch {best_epoch} with Score: {best_score*100:.2f}%")

    # --- Load best model for evaluation ---
    vit_model.load_state_dict(best_model_state)

    # --- Evaluation ---
    print(f"\n  Evaluating fold {fold+1}...")
    vit_model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for cochleograms, labels in val_loader:
            cochleograms, labels = cochleograms.to(device), labels.to(device)
            outputs = vit_model(cochleograms)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Accumulate for aggregated CM
    all_preds_total.extend(all_preds)
    all_labels_total.extend(all_labels)

    print(f"  True label distribution:      {dict(sorted(Counter(all_labels).items()))}")
    print(f"  Predicted label distribution: {dict(sorted(Counter(all_preds).items()))}")

    # ── Per-fold metric block ──
    all_preds_arr  = np.array(all_preds)
    all_labels_arr = np.array(all_labels)

    TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
    FN = np.sum((all_labels_arr != 0) & (all_preds_arr == 0))                                                      # paper def: adventitious → normal
    FN_wrong_type = np.sum((all_labels_arr != 0) & (all_preds_arr != 0) & (all_preds_arr != all_labels_arr))      # subtype confusion, stored only
    TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
    FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

    assert FN + FN_wrong_type + TP == np.sum(all_labels_arr != 0), "Fold adventitious decomposition mismatch"

    # Paper convention: binary normal-vs-adventitious; positives = ALL adventitious
    # flagged adventitious (correct OR wrong subtype) = TP + FN_wrong_type.
    TP_bin = TP + FN_wrong_type
    sensitivity = TP_bin / (TP_bin + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    precision   = TP_bin / (TP_bin + FP + 1e-8)
    accuracy    = (TP_bin + TN) / (TP_bin + TN + FP + FN + 1e-8)
    score       = (sensitivity + specificity) / 2.0

    fold_results.append({
        'fold': fold + 1,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'accuracy': accuracy,
        'score': score,
        # stored for later analysis
        'FN_wrong_type': FN_wrong_type,
    })

    print(f"\n  --- Fold {fold+1} Results ---")
    print(f"  Accuracy:    {accuracy*100:.2f}%")
    print(f"  Sensitivity: {sensitivity*100:.2f}%")
    print(f"  Specificity: {specificity*100:.2f}%")
    print(f"  Precision:   {precision*100:.2f}%")
    print(f"  Score:       {score*100:.2f}%")
    print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}  FN_wrong_type={FN_wrong_type}")


# ── Aggregated metrics across ALL folds ──────────────────────────────────────
print("\n" + "="*60)
print("AGGREGATED 10-FOLD RESULTS")
print("="*60)

all_preds_arr  = np.array(all_preds_total)
all_labels_arr = np.array(all_labels_total)

TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
FN = np.sum((all_labels_arr != 0) & (all_preds_arr == 0))                                                      # paper def: adventitious → normal
FN_wrong_type = np.sum((all_labels_arr != 0) & (all_preds_arr != 0) & (all_preds_arr != all_labels_arr))      # subtype confusion, stored only
TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

assert FN + FN_wrong_type + TP == np.sum(all_labels_arr != 0), "Aggregated adventitious decomposition mismatch"

# Paper convention: binary normal-vs-adventitious (positives = TP + FN_wrong_type).
TP_bin = TP + FN_wrong_type
sensitivity = TP_bin / (TP_bin + FN + 1e-8)
specificity = TN / (TN + FP + 1e-8)
precision   = TP_bin / (TP_bin + FP + 1e-8)
accuracy    = (TP_bin + TN) / (TP_bin + TN + FP + FN + 1e-8)
score       = (sensitivity + specificity) / 2.0

print(f"  Accuracy:    {accuracy*100:.2f}%")
print(f"  Sensitivity: {sensitivity*100:.2f}%")
print(f"  Specificity: {specificity*100:.2f}%")
print(f"  Precision:   {precision*100:.2f}%")
print(f"  Score:       {score*100:.2f}%")
print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}  FN_wrong_type={FN_wrong_type}")

# ── Per-class metrics (one-vs-rest) ──────────────────────────────────────────
# Note: this section uses the 4-class confusion matrix directly so no changes needed here
print("\n" + "="*60)
print("PER-CLASS RESULTS (One-vs-Rest)")
print("="*60)

class_names = ['Normal', 'Crackles', 'Wheezes', 'Both']
agg_cm_4class = confusion_matrix(all_labels_total, all_preds_total, labels=list(range(4)))
print("\n  4-Class Confusion Matrix:")
print(f"  {'':12}", end="")
for name in class_names:
    print(f"  {name:<10}", end="")
print()
for i, name in enumerate(class_names):
    print(f"  {name:<12}", end="")
    for j in range(4):
        print(f"  {agg_cm_4class[i,j]:<10}", end="")
    print()

for c in range(4):
    TP_c = agg_cm_4class[c, c]
    FN_c = agg_cm_4class[c, :].sum() - TP_c
    FP_c = agg_cm_4class[:, c].sum() - TP_c
    TN_c = agg_cm_4class.sum() - TP_c - FN_c - FP_c

    sen_c = TP_c / (TP_c + FN_c + 1e-8)
    spe_c = TN_c / (TN_c + FP_c + 1e-8)
    pre_c = TP_c / (TP_c + FP_c + 1e-8)
    acc_c = (TP_c + TN_c) / (agg_cm_4class.sum() + 1e-8)
    sco_c = (sen_c + spe_c) / 2.0

    print(f"\n  [{class_names[c]}]")
    print(f"    Sensitivity: {sen_c*100:.2f}%")
    print(f"    Specificity: {spe_c*100:.2f}%")
    print(f"    Precision:   {pre_c*100:.2f}%")
    print(f"    Accuracy:    {acc_c*100:.2f}%")
    print(f"    Score:       {sco_c*100:.2f}%")

print("\n" + "="*60)
print("Cross-validation training finished.")
print("="*60)

Using device: cuda

Class weights (softened ^0.25, renormalized):
  Normal   (0): 0.7628
  Crackles (1): 0.9018
  Wheezes  (2): 1.0861
  Both     (3): 1.2494

FOLD 1/10
  Train samples: 6207 | Val samples: 691
  Train class distribution: {0: 3431, 1: 1518, 2: 812, 3: 446}
[CochleogramViT] Parameters — total: 13,040,644  trainable: 13,040,644

  Epoch    Train Loss     Val Loss       LR           Status
  ------------------------------------------------------------


  1        1.3533         1.4830         2.50e-05     Se= 87.7 Sp= 45.5 Score= 66.6  <- best


  2        1.3165         1.3030         5.00e-05     Se=  0.0 Sp= 98.6 Score= 49.3


  3        1.2560         1.1426         7.50e-05     Se= 71.0 Sp= 69.7 Score= 70.4  <- best


  4        1.2348         1.3731         1.00e-04     Se= 14.4 Sp= 92.9 Score= 53.6


...

  26       0.7811         1.2985         8.85e-06     Se= 59.5 Sp= 57.5 Score= 58.5


  27       0.7646         1.2376         5.73e-06     Se= 48.9 Sp= 66.4 Score= 57.6


  28       0.7402         1.3215         3.25e-06     Se= 57.3 Sp= 59.7 Score= 58.5


  29       0.7618         1.3333         1.45e-06     Se= 58.1 Sp= 59.3 Score= 58.7


  30       0.7337         1.3511         3.65e-07     Se= 59.0 Sp= 58.0 Score= 58.5

  Best checkpoint at epoch 1 with Score: 64.28%

  Evaluating fold 10...
  True label distribution:      {np.int64(0): 464, np.int64(1): 155, np.int64(2): 49, np.int64(3): 23}
  Predicted label distribution: {np.int64(0): 233, np.int64(1): 458}

  --- Fold 10 Results ---
  Accuracy:    57.02%
  Sensitivity: 85.46%
  Specificity: 43.10%
  Precision:   42.36%
  Score:       64.28%
  TP=135  FN=33  TN=200  FP=264  FN_wrong_type=59

AGGREGATED 10-FOLD RESULTS
  Accuracy:    62.13%
  Sensitivity: 66.89%
  Specificity: 57.88%
  Precision:   58.67%
  Score:       62.39%
  TP=1241  FN=1078  TN=2108  FP=1534  FN_wrong_type=937

PER-CLASS RESULTS (One-vs-Rest)

  4-Class Confusion Matrix:
                Normal      Crackles    Wheezes     Both      
  Normal        2108        1030        355         149       
  Crackles      565         1046        189         64        
  Wheezes       339         290       

In [14]:
# ── Strict Sensitivity per paper definition ───────────────────────────────────

print("\n" + "="*60)
print("STRICT SENSITIVITY (TP = correct adventitious class)")
print("="*60)

all_preds_arr  = np.array(all_preds_total)
all_labels_arr = np.array(all_labels_total)

# Reconstruct fold boundaries
fold_sizes = []
for _, val_idx in gkf.split(metadata, groups=groups):
    fold_sizes.append(len(val_idx))

print(f"\n  {'Fold':<6} {'Sensitivity':>13} {'Specificity':>13} {'Score':>10}")
print(f"  {'-'*46}")

cursor = 0
fold_scores = []
for fold_idx, size in enumerate(fold_sizes):
    fold_preds  = all_preds_arr[cursor:cursor + size]
    fold_labels = all_labels_arr[cursor:cursor + size]
    cursor += size

    # TP: adventitious correctly classified (exact match)
    TP = np.sum((fold_labels != 0) & (fold_preds == fold_labels))
    # FN: adventitious predicted as anything other than correct class
    FN = np.sum((fold_labels != 0) & (fold_preds != fold_labels))
    # TN: normal correctly classified as Normal
    TN = np.sum((fold_labels == 0) & (fold_preds == 0))
    # FP: normal incorrectly classified as adventitious
    FP = np.sum((fold_labels == 0) & (fold_preds != 0))

    sensitivity = TP / (TP + FN + 1e-8)
    specificity = TN / (TN + FP + 1e-8)
    score       = (sensitivity + specificity) / 2.0
    fold_scores.append(score)

    print(f"  Fold {fold_idx+1:<2}"
          f"  {sensitivity*100:>11.2f}%"
          f"  {specificity*100:>11.2f}%"
          f"  {score*100:>8.2f}%")

# Aggregated
print(f"\n  {'-'*46}")
TP = np.sum((all_labels_arr != 0) & (all_preds_arr == all_labels_arr))
FN = np.sum((all_labels_arr != 0) & (all_preds_arr != all_labels_arr))
TN = np.sum((all_labels_arr == 0) & (all_preds_arr == 0))
FP = np.sum((all_labels_arr == 0) & (all_preds_arr != 0))

sensitivity = TP / (TP + FN + 1e-8)
specificity = TN / (TN + FP + 1e-8)
precision   = TP / (TP + FP + 1e-8)
accuracy    = (TP + TN) / (TP + TN + FP + FN + 1e-8)
score       = (sensitivity + specificity) / 2.0

print(f"  {'AGG':<6}"
      f"  {sensitivity*100:>11.2f}%"
      f"  {specificity*100:>11.2f}%"
      f"  {score*100:>8.2f}%")

print(f"\n  Accuracy:    {accuracy*100:.2f}%")
print(f"  Precision:   {precision*100:.2f}%")
print(f"  TP={TP}  FN={FN}  TN={TN}  FP={FP}")
print("="*60)


STRICT SENSITIVITY (TP = correct adventitious class)

  Fold     Sensitivity   Specificity      Score
  ----------------------------------------------
  Fold 1         51.46%        69.67%     60.56%
  Fold 2         43.66%        72.34%     58.00%
  Fold 3         23.98%        82.12%     53.05%
  Fold 4         41.56%        42.04%     41.80%
  Fold 5         34.36%        31.08%     32.72%
  Fold 6         23.86%        73.37%     48.62%
  Fold 7         43.05%        67.94%     55.49%
  Fold 8         29.01%        67.91%     48.46%
  Fold 9         37.67%        52.32%     45.00%
  Fold 10        59.47%        43.10%     51.29%

  ----------------------------------------------
  AGG           38.11%        57.88%     48.00%

  Accuracy:    48.55%
  Precision:   44.72%
  TP=1241  FN=2015  TN=2108  FP=1534
